# Airbnb scraping: extra price features

**Goal:** the leakage-free model (grouped CV) reaches R² ≈ 0.74. This notebook collects features the project data lacks
(property type, amenity count, host status, rating, reviews) for a **small sample** of listings per city, then tests whether they explain price beyond what we already have.

**Responsible use:** Airbnb's terms restrict automated access. This is a small, slow, academic sample (about 40 listings per city,
2-4 s delays, no logins, no personal data kept). Check the current terms and robots.txt yourself, and don't scale it up.
The scraped file can't be joined row-by-row to the Kaggle CSVs (no shared listing ID), so it is analysed on its own (last section).

In [13]:
import re, time, random, os
import pandas as pd, numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ---------------- CONFIG ----------------
CITIES = ["Amsterdam", "Athens", "Barcelona", "Berlin", "Budapest",
          "Lisbon", "London", "Paris", "Rome", "Vienna"]
PER_CITY   = 40                       # listings per city (keep small)
CHECK_IN, CHECK_OUT = "2026-11-10", "2026-11-12"   # Tue-Thu = a weekday stay (use Fri-Sun for weekend)
NIGHTS     = 2
OUT_CSV    = "DATA/Airbnb_Scraped_Data.csv"
HEADLESS   = False                    # set True once it works; False lets you see/solve blocks

## 1. Parsing helpers (pure functions, easy to test)

In [14]:
CUR = r"[€$£]"

def parse_price(text, nights=NIGHTS):
    """Return nightly price from booking-sidebar text. Handles '€120 night' and '€240 total'."""
    t = " ".join(text.split())
    m = re.search(rf"({CUR})\s?([\d,\.]+)\s*(?:EUR|USD|GBP)?\s*(night|total)", t, re.I)
    if not m:
        m2 = re.search(rf"({CUR})\s?([\d,]+)", t)          # fallback: first amount
        if not m2: return None, None
        return float(m2.group(2).replace(",", "")), m2.group(1)
    amount = float(m.group(2).replace(",", ""))
    if m.group(3).lower() == "total":
        amount = amount / nights
    return round(amount, 2), m.group(1)

def parse_overview(text):
    t = " ".join(text.split())
    g = lambda p: (re.search(p, t, re.I).group(1) if re.search(p, t, re.I) else None)
    bedrooms = g(r"(\d+)\s*bedrooms?")
    if bedrooms is None and re.search(r"\bstudio\b", t, re.I): bedrooms = 0
    return {
        "guests":   g(r"(\d+)\s*guests?"),
        "bedrooms": bedrooms,
        "beds":     g(r"(\d+)\s*beds?\b"),
        "baths":    g(r"(\d+(?:\.\d+)?)\s*(?:shared\s+)?baths?"),
    }

def parse_property_type(subtitle):
    """'Entire rental unit in Paris, France' -> 'Entire rental unit'"""
    m = re.match(r"\s*(.+?)\s+in\s+", subtitle.strip(), re.I)
    return m.group(1) if m else None

def parse_rating(text):
    t = " ".join(text.split())
    m = re.search(r"(\d\.\d{1,2})\s*[·•]\s*(\d[\d,]*)\s*reviews?", t, re.I)
    if m: return float(m.group(1)), int(m.group(2).replace(",", ""))
    m = re.search(r"Rated\s+(\d\.\d{1,2})\s+out of 5.*?(\d[\d,]*)\s*reviews?", t, re.I)
    if m: return float(m.group(1)), int(m.group(2).replace(",", ""))
    if re.search(r"\bNew\b", t): return None, 0
    return None, None

def parse_amenity_count(text):
    m = re.search(r"Show all\s+(\d+)\s+amenities", " ".join(text.split()), re.I)
    return int(m.group(1)) if m else None

# ---- offline self-test ----
assert parse_price("€120 night") == (120.0, "€")
assert parse_price("€ 240 total before taxes") == (120.0, "€")
assert parse_property_type("Entire rental unit in Paris, France") == "Entire rental unit"
assert parse_property_type("Private room in Rome, Italy") == "Private room"
assert parse_overview("4 guests · 2 bedrooms · 3 beds · 1.5 baths")["bedrooms"] == "2"
assert parse_overview("2 guests · Studio · 1 bed · 1 bath")["bedrooms"] == 0
assert parse_rating("4.92 · 120 reviews") == (4.92, 120)
assert parse_amenity_count("Show all 35 amenities") == 35
print("parsers OK")

parsers OK


## 2. Browser + scraping functions

In [15]:
def make_driver():
    o = Options()
    if HEADLESS: o.add_argument("--headless=new")
    o.add_argument("--window-size=1400,1000")
    o.add_argument("--lang=en-US")
    o.add_experimental_option("prefs", {"intl.accept_languages": "en-US,en"})
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=o)

def search_url(city):
    return (f"https://www.airbnb.com/s/{city}/homes?query={city}"
            f"&checkin={CHECK_IN}&checkout={CHECK_OUT}&adults=2&currency=EUR")

def dismiss_popups(driver):
    for label in ["Accept all", "Accept", "OK", "Got it"]:
        try:
            driver.find_element(By.XPATH, f"//button[normalize-space()='{label}']").click()
            time.sleep(0.5)
        except Exception:
            pass
    try:  # translation / welcome modal
        driver.find_element(By.CSS_SELECTOR, "button[aria-label='Close']").click()
    except Exception:
        pass

def collect_listing_urls(driver, city, n):
    driver.get(search_url(city))
    wait = WebDriverWait(driver, 15)
    urls, pages = [], 0
    while len(urls) < n and pages < 6:
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/rooms/']")))
        except Exception:
            print(f"  [{city}] no results / blocked - check the browser window")
            break
        dismiss_popups(driver)
        for _ in range(4):                       # scroll to trigger lazy loading
            driver.execute_script("window.scrollBy(0, 1200);"); time.sleep(0.6)
        for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/rooms/']"):
            href = (a.get_attribute("href") or "").split("?")[0]
            if href and href not in urls: urls.append(href)
        try:
            nxt = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
            driver.execute_script("arguments[0].click();", nxt); time.sleep(3); pages += 1
        except Exception:
            break
    return urls[:n]

def scrape_listing(driver, url, city):
    # keep dates so a price is shown
    driver.get(f"{url}?check_in={CHECK_IN}&check_out={CHECK_OUT}&adults=2&currency=EUR")
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
    time.sleep(2); dismiss_popups(driver)
    driver.execute_script("window.scrollBy(0, 700);"); time.sleep(1)

    body = driver.find_element(By.TAG_NAME, "body").get_attribute("textContent")
    h1 = driver.find_element(By.TAG_NAME, "h1").text
    try:    subtitle = driver.find_element(By.TAG_NAME, "h2").text
    except Exception: subtitle = ""
    try:
        sidebar = driver.find_element(By.CSS_SELECTOR, "[data-plugin-in-point-id='BOOK_IT_SIDEBAR']").get_attribute("textContent")
    except Exception:
        sidebar = ""
    price, cur = parse_price(sidebar)
    rating, reviews = parse_rating(body)
    row = {
        "city": city.lower(), "url": url, "title": h1,
        "property_type": parse_property_type(subtitle),
        "price_per_night": price, "currency": cur,
        "superhost": int("Superhost" in body),
        "guest_favorite": int("Guest favorite" in body),
        "rating": rating, "reviews": reviews,
        "amenity_count": parse_amenity_count(body),
        **parse_overview(body),
    }
    return row

## 3. Run (resumable: re-run the cell and it skips listings already saved)

In [16]:
os.makedirs("DATA", exist_ok=True)
done = set(pd.read_csv(OUT_CSV)["url"]) if os.path.exists(OUT_CSV) else set()
driver = make_driver()
try:
    for i, city in enumerate(CITIES, 1):
        have = (pd.read_csv(OUT_CSV)["city"].eq(city.lower()).sum() if os.path.exists(OUT_CSV) else 0)
        if have >= PER_CITY: 
            print(f"[{i}/{len(CITIES)}] {city}: already {have}, skipping"); continue
        print(f"\n[{i}/{len(CITIES)}] {city}: collecting links...")
        urls = [u for u in collect_listing_urls(driver, city, PER_CITY + 15) if u not in done]
        ok = have
        for u in urls:
            if ok >= PER_CITY: break
            try:
                row = scrape_listing(driver, u, city)
                if row["price_per_night"] is None:
                    print("   skip (no price)"); continue
                pd.DataFrame([row]).to_csv(OUT_CSV, mode="a", index=False, header=not os.path.exists(OUT_CSV))
                done.add(u); ok += 1
                print(f"   {ok:>3}  {row['price_per_night']}{row['currency']}  {row['property_type']}  "
                      f"amen={row['amenity_count']}  SH={row['superhost']}  ★{row['rating']} ({row['reviews']})")
            except Exception as e:
                print("   skip:", type(e).__name__)
            time.sleep(random.uniform(2, 4))        # be polite
finally:
    driver.quit()


[1/10] Amsterdam: collecting links...
   skip (no price)
   skip (no price)
     4  185.0€  Hotel  amen=11  SH=1  ★4.77 (1685)
     5  136.0€  Hotel  amen=11  SH=1  ★4.67 (5098)
     6  264.0€  Hotel  amen=3  SH=1  ★4.44 (18)
     7  35.0€  Hotel  amen=10  SH=1  ★4.83 (2156)
     8  30.0€  Hotel  amen=8  SH=1  ★4.18 (119)
     9  285.0€  Hotel  amen=6  SH=1  ★4.62 (100)
    10  28.0€  Hotel  amen=12  SH=1  ★4.73 (119)
    11  252.0€  Hotel  amen=12  SH=1  ★4.82 (243)
    12  31.0€  Hotel  amen=14  SH=1  ★4.78 (1662)
    13  36.0€  Hotel  amen=11  SH=1  ★4.8 (892)
    14  240.0€  Hotel  amen=8  SH=1  ★4.72 (401)
   skip (no price)
    15  28.0€  Hotel  amen=12  SH=1  ★None (0)
    16  23.0€  Hotel  amen=8  SH=1  ★4.35 (1565)
   skip (no price)
    17  250.0€  Hotel  amen=7  SH=1  ★4.64 (1087)
    18  291.0€  Hotel  amen=11  SH=1  ★4.85 (192)
    19  29.0€  Hotel  amen=10  SH=1  ★4.73 (938)
    20  319.0€  Hotel  amen=10  SH=1  ★4.59 (294)
    21  210.0€  Hotel  amen=10  SH=1  ★4.75 (81

## 4. Load and clean

In [17]:
sc = pd.read_csv(OUT_CSV).drop_duplicates("url")
for c in ["guests", "bedrooms", "beds", "baths"]:
    sc[c] = pd.to_numeric(sc[c], errors="coerce")
sc = sc.dropna(subset=["price_per_night", "guests"])
sc = sc[sc["price_per_night"].between(10, sc["price_per_night"].quantile(0.99))]
sc["log_price"] = np.log1p(sc["price_per_night"])
print(sc.shape); print(sc["city"].value_counts()); sc.head()

(385, 16)
city
amsterdam    40
athens       40
barcelona    40
budapest     40
paris        40
rome         40
vienna       40
lisbon       38
london       38
berlin       29
Name: count, dtype: int64


,city,url,title,property_type,price_per_night,currency,superhost,guest_favorite,rating,reviews,amenity_count,guests,bedrooms,beds,baths,log_price
0,amsterdam,https://www.airbnb.com/rooms/1454892558070097700,The Social Hub Amsterdam West 4*,Hotel,170.0,€,1,0,4.77,797.0,9.0,2,0.0,1.0,NaN,5.141664
1,amsterdam,https://www.airbnb.com/rooms/1142549760112481397,Urban Lodge Hotel,Hotel,23.0,€,1,0,4.75,1801.0,11.0,2,NaN,2.0,NaN,3.178054
2,amsterdam,https://www.airbnb.com/rooms/1591948064662538384,Hotel Nes,Hotel,258.0,€,1,0,4.58,290.0,10.0,2,NaN,1.0,NaN,5.556828
3,athens,https://www.airbnb.com/rooms/1330983501559790704,Room with Unmatched Acropolis Views from Terrace,Room,81.0,€,1,0,4.84,128.0,28.0,2026,1.0,NaN,14.0,4.406719
4,athens,https://www.airbnb.com/rooms/37249317,Relaxing small studio near to Acropolis 3,Entire rental unit,22.0,€,1,0,4.72,440.0,23.0,2,1.0,1.0,1.0,3.135494


## 5. Do the extra features add anything?

The scraped sample can't be merged into the Kaggle rows, so test it directly. Compare a **baseline** (what the Kaggle data
also has: city, guests, bedrooms) against the **extended** model (adds property type, amenities, host status, rating, reviews).
Grouped by city-independent listing, using cross-validation. If the extended R² is clearly higher, those features are worth collecting for the full analysis.

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

y = sc["log_price"]
base_num, base_cat = ["guests", "bedrooms"], ["city"]
ext_num = base_num + ["amenity_count", "superhost", "guest_favorite", "rating", "reviews", "baths", "beds"]
ext_cat = base_cat + ["property_type"]

def make(num, cat):
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    return Pipeline([("pre", pre), ("m", RandomForestRegressor(300, random_state=42, n_jobs=-1))])

cv = KFold(5, shuffle=True, random_state=42)       # each listing appears once here, so no weekday/weekend leakage
for name, num, cat in [("baseline (city, guests, bedrooms)", base_num, base_cat),
                       ("extended (+ amenities, type, host, rating...)", ext_num, ext_cat)]:
    r2 = cross_val_score(make(num, cat), sc[num + cat], y, cv=cv, scoring="r2").mean()
    print(f"{name:50s} R² = {r2:.3f}")

# which new features matter?
m = make(ext_num, ext_cat).fit(sc[ext_num + ext_cat], y)
names = m.named_steps["pre"].get_feature_names_out()
imp = pd.Series(m.named_steps["m"].feature_importances_, names).sort_values(ascending=False)
imp.head(10)

baseline (city, guests, bedrooms)                  R² = 0.221
extended (+ amenities, type, host, rating...)      R² = 0.338


num__reviews                0.177544
cat__property_type_Hotel    0.162123
num__rating                 0.158405
num__amenity_count          0.137850
cat__city_athens            0.067064
num__beds                   0.041367
num__baths                  0.037592
cat__city_budapest          0.033261
cat__city_paris             0.027874
num__bedrooms               0.026846
dtype: float64

In [21]:
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# redefined make(): add_indicator=True adds a "was missing" flag for each numeric column
def make(num, cat):
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median", add_indicator=True), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    return Pipeline([("pre", pre),
                     ("m", RandomForestRegressor(300, random_state=42, n_jobs=-1))])

rcv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)   # same splits for both models
b = cross_val_score(make(base_num, base_cat), sc[base_num + base_cat], y, cv=rcv, scoring="r2")
e = cross_val_score(make(ext_num, ext_cat),  sc[ext_num + ext_cat],  y, cv=rcv, scoring="r2")
d = e - b
print(f"baseline {b.mean():.3f} | extended {e.mean():.3f}")
print(f"paired gain {d.mean():.3f} ± {d.std():.3f}, extended better in {(d > 0).mean():.0%} of folds")

baseline 0.259 | extended 0.305
paired gain 0.046 ± 0.104, extended better in 68% of folds


In [22]:
mid_num, mid_cat = base_num, base_cat + ["property_type"]
m_ = cross_val_score(make(mid_num, mid_cat), sc[mid_num + mid_cat], y, cv=rcv, scoring="r2")
print(f"baseline {b.mean():.3f} | + property_type {m_.mean():.3f} | full extended {e.mean():.3f}")

baseline 0.259 | + property_type 0.249 | full extended 0.305
